# CARec encoder and dimension study

- Load the saved Amazon Electronics catalogue, interaction partitions, and ID maps.
- Use the `NI-PML benchmarks` kernel from the project `.venv`.
- Data exploration, text preparation, sampling, splitting, and saving belong in [data_exploration.ipynb](data_exploration.ipynb).


In [15]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError(
        "Select the 'NI-PML benchmarks' kernel or .venv/bin/python, then restart and run the notebook. "
        f"Current Python: {sys.executable}"
    )

PREPARED_DIR = PROJECT_ROOT / "data/processed/electronics_carec_035_seed2023"
if not (PREPARED_DIR / "manifest.json").is_file():
    raise FileNotFoundError("Prepare the data in data_exploration.ipynb first.")
print(f"Prepared dataset: {PREPARED_DIR}")


Prepared dataset: /Users/matouskovar/FIT/NI-PML/data/processed/electronics_carec_035_seed2023


## Read the dataset manifest

- Read the saved configuration and file information.
- Require a verified dataset before loading it.


In [16]:
import gzip
import hashlib
import json

import pandas as pd
from IPython.display import display

manifest_path = PREPARED_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text())

if manifest["status"] != "verified":
    raise ValueError("Prepare and verify the dataset before loading it.")


## Check the saved files

- Compare each file's SHA-256 checksum and size with the manifest.


In [17]:
partition_names = ("train", "valid", "test", "cold")
files_to_load = [
    "items.jsonl.gz",
    "user_ids.tsv.gz",
    "item_ids.tsv.gz",
    "train.tsv.gz",
    "valid.tsv.gz",
    "test.tsv.gz",
    "cold.tsv.gz",
]

for file_name in files_to_load:
    file_path = PREPARED_DIR / file_name
    expected = manifest["files"][file_name]

    with file_path.open("rb") as source:
        actual_hash = hashlib.file_digest(source, "sha256").hexdigest()
    actual_size = file_path.stat().st_size

    assert actual_hash == expected["sha256"], file_name
    assert actual_size == expected["bytes"], file_name


## Load products and ID maps

- Read one product record per JSONL line.
- Load user and product ID maps, keeping their IDs as strings.


In [18]:
items_path = PREPARED_DIR / "items.jsonl.gz"
product_records = []

with gzip.open(items_path, "rt", encoding="utf-8") as source:
    for line in source:
        product = json.loads(line)
        product_records.append(product)

items = pd.DataFrame(product_records)
del product_records

user_map_path = PREPARED_DIR / "user_ids.tsv.gz"
user_map = pd.read_csv(user_map_path, sep="\t", dtype={"user_id": str})

item_map_path = PREPARED_DIR / "item_ids.tsv.gz"
item_map = pd.read_csv(item_map_path, sep="\t", dtype={"item_id": str})


## Load interaction partitions

- Load train, validation, test, and cold interactions.
- Keep user and product IDs as strings and preserve float precision.
- Check row counts against the manifest.


In [19]:
interaction_types = {
    "user_id:token": str,
    "item_id:token": str,
}
partitions = {}

for name in partition_names:
    partition_path = PREPARED_DIR / f"{name}.tsv.gz"
    frame = pd.read_csv(
        partition_path,
        sep="\t",
        dtype=interaction_types,
        float_precision="round_trip",
    )
    expected_rows = manifest["counts"][name]["rows"]
    assert len(frame) == expected_rows, name
    partitions[name] = frame

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


## Show the loaded data

- Print product and user counts and the saved partition counts.
- Show two product texts.
- Recorded protocol findings remain available in `manifest["audit"]`.


In [20]:
print(f"Loaded {len(items):,} products and {len(user_map):,} users")

partition_summary = pd.DataFrame(manifest["counts"]).T
print(partition_summary.to_string())

text_preview = items[["item_id:token", "text"]].head(2)
display(text_preview)


Loaded 34,220 products and 81,520 users
         rows  users  products
train  452606  81520     32416
valid   14853  12386      9322
test   133274  81520     28515
cold    23163  18680      1797


,item_id:token,text
0,0545105668,Elephant Run Visit Amazon's Roland Smith Page ...
1,0789743035,My MacBook Visit Amazon's John Ray Page Compu...
